In [ ]:
import os
import sys
from pathlib import Path
import datetime 

import numpy as np
import pandas as pd
from astropy.table import Table, join, hstack, vstack
from astropy.coordinates import SkyCoord
import astropy.units as u
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic
import matplotlib.dates as md
from scipy.integrate import trapezoid

from utils import better_step
from matplotlib.gridspec import GridSpec
from statsmodels.stats.proportion import proportion_confint
import seaborn as sns
import desispec.io
from desispec.coaddition import coadd_cameras
import redrock.templates
import astropy
from utils import airtovac, get_kernel, smooth_data

# If not using the desiconda version of prospect: EDIT THIS to your path
sys.path.insert(0,"/global/homes/b/bid13/DESI/DESI-II/prospect/py/")
from prospect import viewer, utilities

In [ ]:
# figure defaults for AASTEX AJ

COLUMN_WIDTH = 242.26653/72.27 #in inches
TEXT_WIDTH = 513.11743/72.27
SMALL_SIZE = 9 # in pts
NORMAL_SIZE = 10 
BIG_SIZE = 12
FONT_FAMILY = 'Nimbus Roman No9 L'



params = {
    "font.family": FONT_FAMILY,
    "font.size": NORMAL_SIZE,
    "axes.titlesize": NORMAL_SIZE,
    "axes.labelsize": NORMAL_SIZE,
    
    "xtick.labelsize": SMALL_SIZE,
    "ytick.labelsize": SMALL_SIZE,
    "xtick.top": True,
    "ytick.right": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": NORMAL_SIZE,
    
    "figure.facecolor": "w",
    "figure.dpi": 300,
    'mathtext.fontset' : "cm"
    
}

plt.rcParams.update(params)

In [ ]:
base_path = Path(os.environ["CFS"]) / "desi" / "users" / "bid13" / "DESI_II"
deep_path = base_path / "other_surveys" / "DEEP23"

In [ ]:
hsc_groth_cat = Table.read(deep_path / "HSC_GROTH.fits").to_pandas()
deep_spec_cat = Table.read(deep_path / "alldeep.egs.2012jun13.fits.gz").to_pandas()
deep_spec_cat = deep_spec_cat[deep_spec_cat["ZQUALITY"]>0]
deep_spec_cat = deep_spec_cat[deep_spec_cat["Z"]>0.001]

zcosmos_spec_cat = Table.read(base_path / "other_surveys" / "ZCOSMOS" / "zcosmos-bright.fits").to_pandas()
zcosmos_spec_cat = zcosmos_spec_cat[zcosmos_spec_cat["REDSHIFT"]>0.01]



cat_desi = Table.read(base_path / "pilot_obs" / "MERGED" / "merged_cat_LSST_WL_Y1.fits" )

cat_desi["success"] = (cat_desi["VI_quality"]>2)
cat_desi = vstack([cat_desi[~cat_desi["success"]],
                      cat_desi[cat_desi["success"] & (cat_desi["Z"]>0.001)]])


In [ ]:
data_path = base_path / "pilot_obs" / "MERGED"
spectra = desispec.io.read_spectra(data_path / "spectra_LSST_WL_Y1.fits")
rr_table = Table.read(data_path / "zbest_LSST_WL_Y1.fits")
rr_details = Table.read(data_path / "redrock_details_LSST_WL_Y1.fits")

In [ ]:
templates = dict()
for filename in redrock.templates.find_templates():
    t = redrock.templates.Template(filename)
    templates[(t.template_type, t.sub_type)] = t

In [ ]:
def flux_to_mag(flux):
    return -2.5*np.log10(flux*1e-9) + 8.90

In [ ]:
def process_hsc_cat(hsc_cat):
    hsc_cat["mag_i"] = flux_to_mag(hsc_cat["i_cmodel_flux"])-hsc_cat["a_i"]
    hsc_cat["mag_g"] = flux_to_mag(hsc_cat["g_cmodel_flux"])-hsc_cat["a_g"]
    hsc_cat["mag_r"] = flux_to_mag(hsc_cat["r_cmodel_flux"])-hsc_cat["a_r"]
    hsc_cat["mag_z"] = flux_to_mag(hsc_cat["z_cmodel_flux"])-hsc_cat["a_z"]
    hsc_cat["mag_y"] = flux_to_mag(hsc_cat["y_cmodel_flux"])-hsc_cat["a_y"]

    hsc_cat["mag_i_fiber"] = flux_to_mag(hsc_cat["i_fiber_flux"])-hsc_cat["a_i"]
    
    ## Quality cuts
    # valid I-band flux
    qmask = np.isfinite(hsc_cat["i_cmodel_flux"]) & (hsc_cat["i_cmodel_flux"]>0)
    #cmodel fit not failed
    qmask &= (~hsc_cat["i_cmodel_flag"].values)
    #General Failure Flag
    qmask &= (~hsc_cat["i_sdsscentroid_flag"].values)

    # Possible cuts: Bright objects nearby, bad pixels

    #star-galaxy separation (is point source in I band)
    # mask &= (hsc_cat["i_extendedness_value"]>0)


    i_min = 22
    i_max = 24.5
    mask = (hsc_cat["mag_i"] <i_max) & (hsc_cat["mag_i"] >i_min)
    i_fiber_min = 22
    i_fiber_max = 24.75
    mask &= (hsc_cat["mag_i_fiber"] <i_fiber_max) & (hsc_cat["mag_i_fiber"] >i_fiber_min)
    
    return hsc_cat[qmask & mask]

In [ ]:
hsc_groth_cat = process_hsc_cat(hsc_groth_cat)
hsc_coord = SkyCoord(ra=hsc_groth_cat["ra"].values*u.degree, dec=hsc_groth_cat["dec"].values*u.degree)
deep_coord = SkyCoord(ra=deep_spec_cat["RA"].values*u.degree, dec=deep_spec_cat["DEC"].values*u.degree)

idx, d2d, d3d = deep_coord.match_to_catalog_sky(hsc_coord)
phot_cat = hsc_groth_cat.iloc[idx]

idx_mask = d2d.arcsec<1


cat_deep = pd.concat([deep_spec_cat.reset_index(),phot_cat.reset_index()], axis=1)

cat_deep = cat_deep[idx_mask]
print(len(cat_deep)/len(deep_coord))

In [ ]:
hsc_cosmos_cat = process_hsc_cat(Table.read(base_path / "target_data" / "HSC_COSMOS_I_mag_lim_24.8.fits").to_pandas())
hsc_coord = SkyCoord(ra=hsc_cosmos_cat["ra"].values*u.degree, dec=hsc_cosmos_cat["dec"].values*u.degree)
zcosmos_coord = SkyCoord(ra=zcosmos_spec_cat["RAJ2000"].values*u.degree, dec=zcosmos_spec_cat["DEJ2000"].values*u.degree)

idx, d2d, d3d = zcosmos_coord.match_to_catalog_sky(hsc_coord)
phot_cat = hsc_cosmos_cat.iloc[idx]

idx_mask = d2d.arcsec<1


cat_zcosmos = pd.concat([zcosmos_spec_cat.reset_index(),phot_cat.reset_index()], axis=1)

cat_zcosmos = cat_zcosmos[idx_mask]
print(len(cat_zcosmos)/len(zcosmos_coord))

In [ ]:
cat_deep["success"] = cat_deep["ZQUALITY"]>2
###Figure out how to merge VI spectypes
cat_deep = cat_deep[cat_deep["Z"]>0.001]
cat_zcosmos["success"] = np.isin(cat_zcosmos["CC"].astype(int), [3,4,13,14,23,24,213,214])
cat_zcosmos = cat_zcosmos[cat_zcosmos["REDSHIFT"]>0.01]

In [ ]:
def make_success_plot(cat,colname="mag_i", nbins=10, ref_sample=None, ax =None,range=None, **kwargs):
    success, bin_edges , _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="mean",range=range)
    sums, bin_edges , _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="sum",range=range)
    count, bin_edges, _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="count",range=range)
    # print(sums)
    # print(count)
    adjustment =1.0
    if ref_sample is not None:
        for i in range(len(bin_edges)-1):
            sel_cat = ref_sample[(ref_sample["HSC_i_MAG"]>=bin_edges[i]) & (ref_sample["HSC_i_MAG"]<bin_edges[i])]
            frac_greater = np.sum(sel_cat["Z"]>1.6)/len(sel_cat)
            print(frac_greater)
    # e_success = np.sqrt(success*(1-success)/(count*adjustment))
    # e_success = 1.96 * e_success # 95% CI
    ci_lo, ci_upp = proportion_confint(sums,count,method="beta")
    
    ax = better_step(bin_edges, success, yerr=(ci_lo,ci_upp), ax = ax,**kwargs)
    # ax.set_ylim(0.3,1)
    # ax.grid(linestyle="--")
    return ax

In [ ]:
with_hercules = False
cat_use = cat_desi.copy()

if  not with_hercules:
    cat_desi = cat_use[cat_use["FIELD_NAME"]!="HERCULES"]

# Compare with Deep 2/3

In [ ]:
cat_deep["g-r"] = cat_deep["mag_g"] - cat_deep["mag_r"]
cat_deep["r-z"] = cat_deep["mag_r"] - cat_deep["mag_z"]
cat_deep["i-y"] = cat_deep["mag_i"] - cat_deep["mag_y"]

cat_desi["g-r"] = cat_desi["mag_g"] - cat_desi["mag_r"]
cat_desi["r-z"] = cat_desi["mag_r"] - cat_desi["mag_z"]
cat_desi["i-y"] = cat_desi["mag_i"] - cat_desi["mag_y"]

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(TEXT_WIDTH, 0.5*TEXT_WIDTH))

cat_deep2 = cat_deep.sample(len(cat_desi))
# ax[0].scatter(cat_deep2["r-z"][cat_deep2["success"]], cat_deep2["g-r"][cat_deep2["success"]],marker=".",s=1,alpha=0.1)
# ax[0].scatter(cat_deep2["r-z"][~cat_deep2["success"]], cat_deep2["g-r"][~cat_deep2["success"]],marker=".",s=1)
ax[0].scatter(cat_deep2["i-y"][cat_deep2["success"]], cat_deep2["g-r"][cat_deep2["success"]],marker=".",s=1)
ax[0].scatter(cat_deep2["i-y"][~cat_deep2["success"]], cat_deep2["g-r"][~cat_deep2["success"]],marker=".",s=1,alpha=0.1)
ax[0].set_xlabel("r-z")
ax[0].set_ylabel("g-r")
ax[0].set_xlim(-2.5,2.5)
ax[0].set_ylim(-1,3)
ax[0].set_title("DEEP23")

# ax[1].scatter(cat_desi["r-z"][cat_desi["success"]], cat_desi["g-r"][cat_desi["success"]],marker=".",s=1,alpha=0.1)
# ax[1].scatter(cat_desi["r-z"][~cat_desi["success"]], cat_desi["g-r"][~cat_desi["success"]],marker=".",s=1)

ax[1].scatter(cat_desi["i-y"][cat_desi["success"]], cat_desi["g-r"][cat_desi["success"]],marker=".",s=1)
ax[1].scatter(cat_desi["i-y"][~cat_desi["success"]], cat_desi["g-r"][~cat_desi["success"]],marker=".",s=1,alpha=0.1)
ax[1].set_xlabel("r-z")
ax[1].set_ylabel("g-r")
ax[1].set_xlim(-2.5,2.5)
ax[1].set_ylim(-1,3)
ax[1].set_title("DESI")

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(TEXT_WIDTH, 0.5*TEXT_WIDTH))
c = ax[0].scatter(cat_desi["r-z"][cat_desi["success"]], cat_desi["g-r"][cat_desi["success"]],marker=".",s=1,c=cat_desi["Z"][cat_desi["success"]],vmin=0.01,vmax=1.6)
fig.colorbar(c,)
# ax[0].scatter(cat_desi["r-z"][~cat_desi["success"]], cat_desi["g-r"][~cat_desi["success"]],marker=".",s=1)
ax[0].set_xlabel("r-z")
ax[0].set_ylabel("g-r")
ax[0].set_xlim(-2,3)
ax[0].set_ylim(-1,3)
ax[0].set_title("Z")

c = ax[1].scatter(cat_desi["r-z"][cat_desi["success"]], cat_desi["g-r"][cat_desi["success"]],marker=".",s=1,c=cat_desi["mag_i"][cat_desi["success"]],vmin=22.5,vmax=24.5)
x = np.linspace(-2,3,100)
y = -1.2*x + 0

plt.plot(x,y,c="k",ls="--")

fig.colorbar(c,)
# ax[1].scatter(cat_desi["r-z"][~cat_desi["success"]], cat_desi["g-r"][~cat_desi["success"]],marker=".",s=1)
ax[1].set_xlabel("r-z")
ax[1].set_ylabel("g-r")
ax[1].set_xlim(-2,3)
ax[1].set_ylim(-1,3)
ax[1].set_title("i-mag")

## Plot some spectra to chose the best ones

In [ ]:
rr_table[rr_table["SUBTYPE"].mask]["SUBTYPE"] = b'NA'
rr_table["SUBTYPE"].mask =False

for c in rr_table.columns:
    # print(c)
    if isinstance(rr_table[c], astropy.table.column.MaskedColumn):
        rr_table[c] = rr_table[c].value.data
rr_table.add_index("TARGETID")


for c in rr_details.columns:
    if c == "SPECTYPE":
        rr_details[c] = rr_details[c].value.astype(str)
    if c =="SUBTYPE":
        rr_details[c] = rr_details[c].value.data.astype(str)
rr_details.add_index("TARGETID")




cat_desi.add_index("TARGETID")




In [ ]:
# mask_color = (cat_desi["g-r"] > -1*cat_desi["r-z"] +3) 
# sel_mask =   (cat_desi["success"]) & mask_color  & (cat_desi["mag_i"]>24) & (cat_desi["DELTACHI2"]<100) # & (cat_desi["i-y"]>1) #& (cat_desi["Z"]>1)  
# print(sel_mask.sum())

In [ ]:
sel_targets = [39089837487179397,39089837453634450, 39089837487179278, 39089837453631567]


# sel_targets = cat_desi["TARGETID"][sel_mask].value
sel_spectra = spectra.select(targets = sel_targets)

# sel_spectra.fibermap = join(sel_spectra.fibermap, cat[['mag_i','mag_i_fiber',"TARGETID"]],keys="TARGETID",join_type="left")
sel_spectra.fibermap['mag_i'] = cat_desi.loc[sel_spectra.fibermap['TARGETID']]['mag_i']
sel_spectra.fibermap['mag_i_fiber'] =  cat_desi.loc[sel_spectra.fibermap['TARGETID']]['mag_i_fiber']
sel_targets = sel_spectra.fibermap["TARGETID"].value

sel_zcat = rr_table.loc[sel_targets]


sel_rr_det = rr_details.loc[sel_targets]


In [ ]:
# red_low_snr = [39089837453631933, ] # 39089837487179397
# red_high_snr =  [39089837453634450] #39089837487179296
# blue_lo = 39089837487179278
#blue_high = 39089837453631567

In [ ]:
# Run this cell to have the VI tool !
viewer.plotspectra(sel_spectra, 
                   zcatalog=sel_zcat,
                   redrock_cat=sel_rr_det,
                   notebook=True, title='test_exptime_gr_7', 
                   html_dir = "/global/cfs/cdirs/desi/users/bid13/DESI_II/VI_pages"
                   # mask_type='SV1_DESI_TARGET'
                  )

In [ ]:
Ca_K = airtovac(3933.7)
Ca_H = airtovac(3968.5)
Ca_G = airtovac(4307.74)

H_alpha = airtovac(6562.801)
H_beta = airtovac(4861.325)
H_delta = airtovac(4101.734)
H_gamma = airtovac(4340.464)

OII_1 = airtovac(3726.032)
OII_2 = airtovac(3728.815)
OIII_1 = airtovac(4958.911)
OIII_2 = airtovac(5006.843)

MgII_1= airtovac(2796.3543)
MgII_2= airtovac(2803.5315)



In [ ]:
fig , ax = plt.subplots(4,1,figsize=(TEXT_WIDTH, TEXT_WIDTH),sharex=True)
spectra_coadd = coadd_cameras(sel_spectra)

target_id_order = [39089837453634450,39089837487179397,39089837453631567,39089837487179278] #high-red,low-red,high-blue,low-blue

ylim = [(-0.1,0.5),(-0.12,0.15),(-0.25,2.1),(-0.15,0.19)]
for i in range(4):
    targetid = target_id_order[i]
    spec_plot = spectra_coadd.select(targets = targetid)
    spec_z = cat_desi.loc[targetid]["Z"]
    mag_i = cat_desi.loc[targetid]["mag_i"]
    exptime = cat_desi.loc[targetid]["EXPTIME"]/60
    flux = np.squeeze(spec_plot.flux["brz"])
    wave = spec_plot.wave["brz"]
    ivar = np.squeeze(spec_plot.ivar["brz"])

    ax[i].plot(wave,flux,lw=1,alpha=0.15,c="C0",zorder=19)
    if i in [0,1]:
        nsmooth = 5
        kernel = get_kernel(nsmooth)
    else:
        nsmooth = 3
        kernel = get_kernel(nsmooth)
    ax[i].plot(wave, smooth_data(flux, kernel, ivar_in=ivar, ivar_weight=True),lw=0.8,c="C0",zorder=20)
    ax[i].set_ylim(*ylim[i])

    ax[i].set_title(f"TARGETID = {targetid}, "+r"$m_{i}$ = "+f"{mag_i:.2f}, Redshift = {spec_z:.3f}, "+r"$T_{\mathrm{exp}}$ = "+f"{exptime:.0f} min")

    vline_lw = 0.8
    vline_ls = "--"
    vline_c = "k"
    vline_alpha = 0.5
    if i in [0,1]:
        ax[i].axvline(Ca_H*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        ax[i].axvline(Ca_K*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        ax[i].axvline(Ca_G*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        ax[i].axvline(H_delta*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        ax[i].axvline(H_gamma*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        ax[i].axvline(MgII_1*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        ax[i].axvline(MgII_2*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        
        if i ==0:
            y_text_offset = -0.03
        if i == 1:
            y_text_offset = -0.09
        ax[i].text(Ca_H*(1+spec_z)+30, y_text_offset, 'H',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1)
        ax[i].text(Ca_K*(1+spec_z)-30, y_text_offset, 'K',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1,ha="right")
        ax[i].text(Ca_G*(1+spec_z)-30, y_text_offset, 'G',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1,ha="right")
        ax[i].text(H_delta*(1+spec_z)+30, y_text_offset, r'H$\delta$',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1)
        ax[i].text(H_gamma*(1+spec_z)+30, y_text_offset, r'H$\gamma$',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1)
        ax[i].text(MgII_1*(1+spec_z) +30 ,y_text_offset,r'Mg II',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1)
        
        
    if i in [2,3]:
        ax[i].axvline(OII_1*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        ax[i].axvline(OII_2*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        ax[i].axvline(OIII_1*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        ax[i].axvline(OIII_2*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        if i ==2:
            y_text_offset = 1.5
            
        if i == 3:
            y_text_offset = -0.11
            ax[i].text(OIII_1*(1+spec_z)-30, y_text_offset, '[O III]',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1,ha="right")
        ax[i].text(OII_1*(1+spec_z)-30, y_text_offset, '[O II]',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1,ha="right")
        
        
        ax[i].text(OIII_2*(1+spec_z)+30, y_text_offset, '[O III]',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1)
        
        
    if i==2:
        ax[i].axvline(H_alpha*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        ax[i].axvline(H_beta*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        ax[i].axvline(H_delta*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)
        ax[i].axvline(H_gamma*(1+spec_z),lw=vline_lw,ls=vline_ls,c=vline_c,alpha=vline_alpha)

        ax[i].text(OIII_1*(1+spec_z)+30, y_text_offset-0.5, '[O III]',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1,ha="left")
        ax[i].text(H_alpha*(1+spec_z)-30, y_text_offset, r'H$\alpha$',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1,ha="right")
        ax[i].text(H_beta*(1+spec_z)-30, y_text_offset, r'H$\beta$',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1,ha="right")
        ax[i].text(H_gamma*(1+spec_z)+30, y_text_offset, r'H$\gamma$',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1)
        ax[i].text(H_delta*(1+spec_z)-30, y_text_offset, r'H$\delta$',fontsize=SMALL_SIZE,alpha=vline_alpha+0.1,ha="right")
ax[3].set_xlabel(r"Wavelength [$\AA$]")
ax[2].set_ylabel(r"Flux [$\times 10^{-17}\mathrm{ergs}.\mathrm{s}^{-1}.\mathrm{cm}^{-2}.\AA^{-1}$]",labelpad=20,fontsize=NORMAL_SIZE+1)
l, b, w, h = ax[2].get_position().bounds
ax[2].yaxis.label.set_position([l, b+0.7,w,h])
plt.savefig("./figs/sample_spectra.pdf",bbox_inches="tight",dpi=300)